#CNN1D Raw Waveform

##10 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Conv1D, MaxPooling1D, GlobalAveragePooling1D,
                                     Input)

# ══════════════════════════════════════════════════════════════
#  CNN 1D — ARQUITECTURA VGG19-style sobre Raw Waveform
#  Replica el patrón VGG19: bloques de Conv repetidos (2-4x)
#  con filtros crecientes (64→128→256→512→512) + MaxPool entre bloques.
#  Adaptado a 1D para procesar waveform directamente.
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 220 500
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")

# --- 3. PROCESAMIENTO RAW WAVEFORM ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), SAMPLES_PER_SEGMENT):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2:
                continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))   # (220500, 1)
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. BLOQUE CONV VGG-STYLE ---
def vgg_block_1d(x, filters, n_convs, kernel_size=9):
    """
    Replica un bloque VGG: N capas Conv seguidas sin pooling interno,
    BN después de cada Conv, pool al final del bloque.
    kernel_size=9 en lugar de 3 (VGG 2D) para capturar contexto temporal 1D.
    """
    for _ in range(n_convs):
        x = Conv1D(filters, kernel_size=kernel_size,
                   padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=4)(x)
    x = Dropout(0.2)(x)
    return x

# --- 5. MODELO VGG19-1D ---
def get_model_vgg19_1d(input_length):
    """
    VGG19 1D adaptado:
      Block 1: 2× Conv64   + Pool → equivalente a VGG19 bloques 1-2
      Block 2: 2× Conv128  + Pool
      Block 3: 4× Conv256  + Pool → bloques 3-4 de VGG19
      Block 4: 4× Conv512  + Pool → bloque 5
      Block 5: 4× Conv512  + Pool → bloque 5b
      GlobalAveragePooling (en lugar de Flatten + FC4096 × 2 de VGG19 original)
      → FC512 → FC256 → Sigmoid
    Total bloques conv: 2+2+4+4+4 = 16 ≈ VGG19 (19 capas con FC)
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque 1 — 64 filtros, stride 64 en primera conv para submuestrear agresivamente
    x = Conv1D(64, kernel_size=512, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, kernel_size=9, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque 2 — 128 filtros
    x = vgg_block_1d(x, 128, n_convs=2, kernel_size=9)

    # Bloque 3 — 256 filtros (4 conv como VGG19)
    x = vgg_block_1d(x, 256, n_convs=4, kernel_size=7)

    # Bloque 4 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=5)

    # Bloque 5 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=3)

    # Pooling global (sustituye FC4096 de VGG19; evita explosión de parámetros en 1D)
    x = GlobalAveragePooling1D()(x)

    # Clasificador
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 6. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION VGG19-1D Raw Waveform ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_vgg19_1d(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=16,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 7. REPORTE ---
print("\n" + "="*60)
print(f"RESULTADOS VGG19-1D Raw Waveform  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión VGG19-1D ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — VGG19-1D Raw Waveform')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
pip install numpy librosa matplotlib seaborn tensorflow scikit-learn vit-keras transformers

##2 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Conv1D, MaxPooling1D, GlobalAveragePooling1D,
                                     Input)

# ══════════════════════════════════════════════════════════════
#  CNN 1D — VGG19-style sobre Raw Waveform
#  Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 2                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO RAW WAVEFORM CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. BLOQUE CONV VGG-STYLE ---
def vgg_block_1d(x, filters, n_convs, kernel_size=9):
    for _ in range(n_convs):
        x = Conv1D(filters, kernel_size=kernel_size,
                   padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=4)(x)
    x = Dropout(0.2)(x)
    return x

# --- 5. MODELO VGG19-1D ---
def get_model_vgg19_1d(input_length):
    """
    VGG19 1D adaptado para segmentos de 2s (44100 muestras).
    Primer Conv con stride=16 para submuestrear de 44100 → ~2756.
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque 1 — stride reducido por la secuencia más corta (2s vs 10s)
    x = Conv1D(64, kernel_size=128, strides=16, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, kernel_size=9, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque 2 — 128 filtros
    x = vgg_block_1d(x, 128, n_convs=2, kernel_size=9)

    # Bloque 3 — 256 filtros (4 conv como VGG19)
    x = vgg_block_1d(x, 256, n_convs=4, kernel_size=7)

    # Bloque 4 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=5)

    # Bloque 5 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=3)

    x = GlobalAveragePooling1D()(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 6. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION VGG19-1D | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_vgg19_1d(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 7. REPORTE ---
print("\n" + "="*60)
print(f"RESULTADOS VGG19-1D | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión VGG19-1D 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — VGG19-1D 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##3 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Conv1D, MaxPooling1D, GlobalAveragePooling1D,
                                     Input)

# ══════════════════════════════════════════════════════════════
#  CNN 1D — VGG19-style sobre Raw Waveform
#  Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 3                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO RAW WAVEFORM CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. BLOQUE CONV VGG-STYLE ---
def vgg_block_1d(x, filters, n_convs, kernel_size=9):
    for _ in range(n_convs):
        x = Conv1D(filters, kernel_size=kernel_size,
                   padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=4)(x)
    x = Dropout(0.2)(x)
    return x

# --- 5. MODELO VGG19-1D ---
def get_model_vgg19_1d(input_length):
    """
    VGG19 1D adaptado para segmentos de 3s (66150 muestras).
    Primer Conv con stride=32 para submuestrear de 66150 → ~2067.
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque 1
    x = Conv1D(64, kernel_size=256, strides=32, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, kernel_size=9, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque 2 — 128 filtros
    x = vgg_block_1d(x, 128, n_convs=2, kernel_size=9)

    # Bloque 3 — 256 filtros (4 conv)
    x = vgg_block_1d(x, 256, n_convs=4, kernel_size=7)

    # Bloque 4 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=5)

    # Bloque 5 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=3)

    x = GlobalAveragePooling1D()(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 6. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION VGG19-1D | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_vgg19_1d(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 7. REPORTE ---
print("\n" + "="*60)
print(f"RESULTADOS VGG19-1D | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión VGG19-1D 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — VGG19-1D 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##4 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Conv1D, MaxPooling1D, GlobalAveragePooling1D,
                                     Input)

# ══════════════════════════════════════════════════════════════
#  CNN 1D — VGG19-style sobre Raw Waveform
#  Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 4                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO RAW WAVEFORM CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. BLOQUE CONV VGG-STYLE ---
def vgg_block_1d(x, filters, n_convs, kernel_size=9):
    for _ in range(n_convs):
        x = Conv1D(filters, kernel_size=kernel_size,
                   padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=4)(x)
    x = Dropout(0.2)(x)
    return x

# --- 5. MODELO VGG19-1D ---
def get_model_vgg19_1d(input_length):
    """
    VGG19 1D adaptado para segmentos de 3s (66150 muestras).
    Primer Conv con stride=32 para submuestrear de 66150 → ~2067.
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque 1
    x = Conv1D(64, kernel_size=256, strides=32, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, kernel_size=9, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque 2 — 128 filtros
    x = vgg_block_1d(x, 128, n_convs=2, kernel_size=9)

    # Bloque 3 — 256 filtros (4 conv)
    x = vgg_block_1d(x, 256, n_convs=4, kernel_size=7)

    # Bloque 4 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=5)

    # Bloque 5 — 512 filtros (4 conv)
    x = vgg_block_1d(x, 512, n_convs=4, kernel_size=3)

    x = GlobalAveragePooling1D()(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 6. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION VGG19-1D | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_vgg19_1d(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 7. REPORTE ---
print("\n" + "="*60)
print(f"RESULTADOS VGG19-1D | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión VGG19-1D 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy',lw=2,linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — VGG19-1D 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

#Rnn Gru

##10 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     GRU, GlobalAveragePooling1D, Input,
                                     Conv1D, MaxPooling1D)

# ══════════════════════════════════════════════════════════════════════
#  RNN — GRU sobre Raw Waveform  (optimizado para velocidad)
#
#  ¿Por qué GRU y no LSTM/BiLSTM?
#  • GRU tiene ~30% menos parámetros que LSTM (2 gates vs 3)
#  • Sin bidireccionalidad → mitad de cómputo por paso temporal
#  • CuDNNGRU de TF usa kernels CUDA optimizados automáticamente
#    si no hay recurrent_dropout → 5-10× más rápido en GPU
#
#  Estrategia para reducir longitud de secuencia:
#  Una CNN 1D ligera primero subsamplea el waveform de 220k muestras
#  a ~430 timesteps antes de entrar a las capas GRU.
#  Esto es la mayor causa de lentitud en BiLSTM: secuencias largas.
# ══════════════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 220 500
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")

# --- 3. PROCESAMIENTO RAW WAVEFORM ---
def process_file(file_path, label):
    """
    Raw waveform → (SAMPLES_PER_SEGMENT, 1).
    La CNN interna hará la reducción de longitud antes de la GRU.
    """
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), SAMPLES_PER_SEGMENT):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2:
                continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))   # (220500, 1)
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + GRU ---
def get_model_rnn(input_length):
    """
    Arquitectura CNN → GRU:

    [1] CNN Subsampler (rápido):
        Tres bloques Conv1D + MaxPool reducen 220500 → ~430 timesteps.
        Esto es CLAVE: la GRU solo procesa 430 pasos en lugar de 220k.
        Aquí reside la diferencia de velocidad vs BiLSTM directo.

    [2] GRU Stack (sin recurrent_dropout):
        TF activa automáticamente CuDNNGRU en GPU cuando:
          - No hay recurrent_dropout
          - activation='tanh', recurrent_activation='sigmoid'
          - unroll=False
        Esto usa kernels CUDA nativos → 5-10× más rápido que LSTM.

    [3] Clasificador ligero.
    """
    inputs = Input(shape=(input_length, 1))

    # ── CNN Subsampler ────────────────────────────────────────────────
    # Bloque 1: stride grande → reduce 220500 → ~860
    x = Conv1D(64, kernel_size=256, strides=128, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)          # → ~430
    x = Dropout(0.2)(x)

    # Bloque 2: extrae features más abstractos
    x = Conv1D(128, kernel_size=9, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Bloque 3: proyección final antes de GRU
    x = Conv1D(128, kernel_size=5, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x shape: (batch, ~430, 128)

    # ── GRU Stack (CuDNN-compatible: sin recurrent_dropout) ──────────
    # Capa 1: captura dinámica temporal de largo alcance
    x = GRU(128, return_sequences=True,
            activation='tanh',
            recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    # Capa 2: refinamiento de la secuencia
    x = GRU(64, return_sequences=True,
            activation='tanh',
            recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    # Capa 3: resumen final de la secuencia
    x = GRU(32, return_sequences=False,   # devuelve solo el último estado
            activation='tanh',
            recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    # ── Clasificador ──────────────────────────────────────────────────
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CNN_GRU_RawWaveform')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION RNN/GRU Raw Waveform ({N_FOLDS} FOLDS)")
print("Arquitectura: CNN Subsampler → GRU Stack (CuDNN-optimizado)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")

    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(y_train),
                                                 y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_rnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5,
                                          restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5,
                                              patience=3, min_lr=1e-7),
    ]

    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    print("Evaluando pacientes...")
    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx:
            continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS GLOBALES RNN/GRU Raw Waveform  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión RNN/GRU ({N_FOLDS}-Fold CV) — Raw Waveform')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)

print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='chocolate', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — RNN/GRU Raw Waveform')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##2 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     GRU, Input, Conv1D, MaxPooling1D)

# ══════════════════════════════════════════════════════════════
#  RNN / GRU — CNN Subsampler + GRU Stack (CuDNN-optimizado)
#  Raw Waveform | Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 2                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + GRU ---
def get_model_rnn(input_length):
    """
    CNN Subsampler: 44100 → ~86 timesteps antes de las GRU.
    GRU CuDNN-compatible: sin recurrent_dropout → kernels CUDA nativos.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler
    x = Conv1D(64, kernel_size=64, strides=32, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=5, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x shape: (batch, ~86, 128)

    # GRU Stack (CuDNN)
    x = GRU(128, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(64, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(32, return_sequences=False,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CNN_GRU_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION RNN/GRU | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_rnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS RNN/GRU | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión RNN/GRU 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='chocolate', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — RNN/GRU 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##3 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     GRU, Input, Conv1D, MaxPooling1D)

# ══════════════════════════════════════════════════════════════
#  RNN / GRU — CNN Subsampler + GRU Stack (CuDNN-optimizado)
#  Raw Waveform | Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 3                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + GRU ---
def get_model_rnn(input_length):
    """
    CNN Subsampler: 66150 → ~129 timesteps antes de las GRU.
    GRU CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler
    x = Conv1D(64, kernel_size=128, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=5, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x shape: (batch, ~129, 128)

    # GRU Stack (CuDNN)
    x = GRU(128, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(64, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(32, return_sequences=False,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CNN_GRU_3s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION RNN/GRU | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_rnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS RNN/GRU | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión RNN/GRU 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='chocolate', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — RNN/GRU 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##4 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     GRU, Input, Conv1D, MaxPooling1D)

# ══════════════════════════════════════════════════════════════
#  RNN / GRU — CNN Subsampler + GRU Stack (CuDNN-optimizado)
#  Raw Waveform | Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 4                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + GRU ---
def get_model_rnn(input_length):
    """
    CNN Subsampler: 66150 → ~129 timesteps antes de las GRU.
    GRU CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler
    x = Conv1D(64, kernel_size=128, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=5, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x shape: (batch, ~129, 128)

    # GRU Stack (CuDNN)
    x = GRU(128, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(64, return_sequences=True,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = GRU(32, return_sequences=False,
            activation='tanh', recurrent_activation='sigmoid')(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CNN_GRU_3s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION RNN/GRU | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_rnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS RNN/GRU | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión RNN/GRU 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='chocolate', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — RNN/GRU 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

#BiLSTM Raw waveform

##tensorflow 10 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, LSTM, GlobalAveragePooling1D,
                                     Conv1D, MaxPooling1D, Input)

# ══════════════════════════════════════════════════════════════════════
#  BiLSTM OPTIMIZADO — Raw Waveform
#
#  Cambios respecto a la versión lenta:
#  ┌──────────────────────────────────────────────────────────────┐
#  │ PROBLEMA ORIGINAL          │ SOLUCIÓN APLICADA               │
#  ├──────────────────────────────────────────────────────────────┤
#  │ Secuencia de 220 500 pasos │ CNN reduce a ~430 pasos ANTES   │
#  │ directo al BiLSTM          │ de entrar al BiLSTM             │
#  ├──────────────────────────────────────────────────────────────┤
#  │ recurrent_dropout=0.1      │ Eliminado → activa CuDNNLSTM    │
#  │ desactiva CuDNN            │ (kernels CUDA nativos, 5-10×)   │
#  ├──────────────────────────────────────────────────────────────┤
#  │ 3 capas BiLSTM profundas   │ 2 capas BiLSTM más pequeñas     │
#  │ (256+128+64 unidades)      │ (128+64 unidades)               │
#  ├──────────────────────────────────────────────────────────────┤
#  │ batch_size=32              │ batch_size=64 → mejor uso GPU   │
#  ├──────────────────────────────────────────────────────────────┤
#  │ Estadísticas manuales por  │ CNN aprende features            │
#  │ frame (lento en CPU)       │ directamente (todo en GPU)      │
#  └──────────────────────────────────────────────────────────────┘
# ══════════════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 10
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 220 500
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")

# --- 3. PROCESAMIENTO RAW WAVEFORM ---
def process_file(file_path, label):
    """
    Raw waveform → (SAMPLES_PER_SEGMENT, 1).
    Sin preprocesamiento manual por frame: la CNN lo hace en GPU.
    """
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio), SAMPLES_PER_SEGMENT):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT // 2:
                continue
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + BiLSTM OPTIMIZADO ---
def get_model_bilstm_fast(input_length):
    """
    CNN Subsampler → BiLSTM (CuDNN-compatible)

    Reglas para activar CuDNNLSTM automático en TF/Keras:
      ✓ activation='tanh'
      ✓ recurrent_activation='sigmoid'
      ✓ recurrent_dropout=0  (NO usar recurrent_dropout)
      ✓ unroll=False
      ✓ use_bias=True
    Con estas condiciones TF usa los kernels CUDA nativos → 5-10× más rápido.
    El Dropout regular entre capas sí está permitido.
    """
    inputs = Input(shape=(input_length, 1))

    # ── CNN Subsampler: 220500 → ~430 timesteps ───────────────────────
    # Stride grande en la primera capa = reducción agresiva en 1 paso
    x = Conv1D(64, kernel_size=256, strides=128,
               padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)          # 220500/128/2 ≈ 860 → 430
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1,
               padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x: (batch, ~430, 128)

    # ── BiLSTM CuDNN-compatible (sin recurrent_dropout) ──────────────
    x = Bidirectional(LSTM(128, return_sequences=True,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(64, return_sequences=False,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    # ── Clasificador ──────────────────────────────────────────────────
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_Fast_RawWaveform')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# # --- 5. CROSS-VALIDATION ---
# skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
# global_y_true, global_y_pred, global_y_prob = [], [], []

# print(f"\nINICIANDO CROSS-VALIDATION BiLSTM RÁPIDO ({N_FOLDS} FOLDS)")
# print("CNN Subsampler 220k→430 pasos + CuDNNLSTM (sin recurrent_dropout)\n")

# for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
#     print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")

#     Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
#     ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
#     Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

#     print("Generando segmentos...")
#     X_train, y_train = build_dataset(Xtr_f, ytr_f)
#     print(f"  Shape train: {X_train.shape}")

#     weights = class_weight.compute_class_weight('balanced',
#                                                  classes=np.unique(y_train),
#                                                  y=y_train)
#     cw = dict(enumerate(weights))

#     model = get_model_bilstm_fast(SAMPLES_PER_SEGMENT)
#     if fold_idx == 0:
#         model.summary()

#     cbs = [
#         tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5,
#                                           restore_best_weights=True),
#         tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5,
#                                               patience=3, min_lr=1e-7),
#     ]

#     model.fit(X_train, y_train, epochs=25, batch_size=64,
#               class_weight=cw, callbacks=cbs, verbose=1)
#     print("Entrenamiento completado.")

#     for i, fp in enumerate(Xte_f):
#         sx, _ = process_file(fp, yte_f[i])
#         if not sx:
#             continue
#         sx = np.array(sx, np.float32)
#         avg_prob = float(np.mean(model.predict(sx, verbose=0)))
#         global_y_true.append(int(yte_f[i]))
#         global_y_pred.append(1 if avg_prob > 0.33 else 0)
#         global_y_prob.append(avg_prob)

#     tf.keras.backend.clear_session()
#     del model, X_train, y_train

# # --- 6. REPORTE FINAL ---
# print("\n" + "="*60)
# print(f"RESULTADOS BiLSTM RÁPIDO Raw Waveform  ({len(global_y_true)} pacientes)")
# print("="*60)

# cm = confusion_matrix(global_y_true, global_y_pred)
# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
#             xticklabels=['No Depresión', 'Depresión'],
#             yticklabels=['No Depresión', 'Depresión'])
# plt.title(f'Confusión BiLSTM Rápido ({N_FOLDS}-Fold CV) — Raw Waveform')
# plt.ylabel('Real'); plt.xlabel('Predicción')
# plt.tight_layout(); plt.show()

# acc      = accuracy_score(global_y_true, global_y_pred)
# prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
# rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
# f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
# f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)

# print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
# print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
# print(classification_report(global_y_true, global_y_pred,
#                              target_names=['No Depresión', 'Depresión']))

# fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
# roc_auc = auc(fpr, tpr)
# plt.figure(figsize=(7, 6))
# plt.plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {roc_auc:.2f}')
# plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
# plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
# plt.title('ROC — BiLSTM Rápido Raw Waveform')
# plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##10 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, LSTM, Conv1D,
                                     MaxPooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  BiLSTM OPTIMIZADO — Raw Waveform
#  Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 10                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + BiLSTM ---
def get_model_bilstm_fast(input_length):
    """
    CNN Subsampler: 44100 → ~86 timesteps antes del BiLSTM.
    CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler: stride=64 → ~689, pool/2 → ~344
    x = Conv1D(64, kernel_size=64, strides=32,
               padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1,
               padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x: (batch, ~172, 128)

    # BiLSTM CuDNN-compatible
    x = Bidirectional(LSTM(128, return_sequences=True,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(64, return_sequences=False,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION BiLSTM | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_bilstm_fast(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=64,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS BiLSTM | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión BiLSTM 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — BiLSTM 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##2 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, LSTM, Conv1D,
                                     MaxPooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  BiLSTM OPTIMIZADO — Raw Waveform
#  Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 2                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + BiLSTM ---
def get_model_bilstm_fast(input_length):
    """
    CNN Subsampler: 44100 → ~86 timesteps antes del BiLSTM.
    CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler: stride=64 → ~689, pool/2 → ~344
    x = Conv1D(64, kernel_size=64, strides=32,
               padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1,
               padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x: (batch, ~172, 128)

    # BiLSTM CuDNN-compatible
    x = Bidirectional(LSTM(128, return_sequences=True,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(64, return_sequences=False,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION BiLSTM | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_bilstm_fast(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=64,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS BiLSTM | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión BiLSTM 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — BiLSTM 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##3 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, LSTM, Conv1D,
                                     MaxPooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  BiLSTM OPTIMIZADO — Raw Waveform
#  Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 3                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + BiLSTM ---
def get_model_bilstm_fast(input_length):
    """
    CNN Subsampler: 66150 → ~258 timesteps antes del BiLSTM.
    CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler
    x = Conv1D(64, kernel_size=128, strides=64,
               padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1,
               padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x: (batch, ~258, 128)

    # BiLSTM CuDNN-compatible
    x = Bidirectional(LSTM(128, return_sequences=True,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(64, return_sequences=False,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_3s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION BiLSTM | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_bilstm_fast(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=64,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS BiLSTM | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión BiLSTM 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — BiLSTM 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 4 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, LSTM, Conv1D,
                                     MaxPooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  BiLSTM OPTIMIZADO — Raw Waveform
#  Segmentos de 4 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 4                                      # ← 4 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 88 200
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 44 100
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(Xf, yf):
    f0, f1 = Xf[yf == 0], Xf[yf == 1]
    diff = len(f0) - len(f1)
    if diff > 0:
        idx = np.random.choice(len(f1), size=diff, replace=True)
        return np.concatenate([Xf, f1[idx]]), np.concatenate([yf, np.ones(diff, int)])
    return Xf, yf

def build_dataset(file_list, label_list):
    X, y = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file(fp, label_list[i])
        X.extend(sx); y.extend(sy)
    return np.array(X, np.float32), np.array(y, np.int32)

# --- 4. MODELO CNN + BiLSTM ---
def get_model_bilstm_fast(input_length):
    """
    CNN Subsampler: 88200 → ~344 timesteps antes del BiLSTM.
    CuDNN-compatible: sin recurrent_dropout.
    """
    inputs = Input(shape=(input_length, 1))

    # CNN Subsampler
    x = Conv1D(64, kernel_size=128, strides=64,
               padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=9, strides=1,
               padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    # x: (batch, ~344, 128)

    # BiLSTM CuDNN-compatible
    x = Bidirectional(LSTM(128, return_sequences=True,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(64, return_sequences=False,
                           activation='tanh',
                           recurrent_activation='sigmoid',
                           recurrent_dropout=0))(x)
    x = Dropout(0.35)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_4s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION BiLSTM | 4s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    Xtr_f, Xte_f = all_files[tr_idx], all_files[te_idx]
    ytr_f, yte_f = all_labels[tr_idx], all_labels[te_idx]
    Xtr_f, ytr_f = oversample_minority(Xtr_f, ytr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(Xtr_f, ytr_f)
    print(f"  Shape train: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_bilstm_fast(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7),
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=64,
              class_weight=cw, callbacks=cbs, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(Xte_f):
        sx, _ = process_file(fp, yte_f[i])
        if not sx: continue
        sx = np.array(sx, np.float32)
        avg_prob = float(np.mean(model.predict(sx, verbose=0)))
        global_y_true.append(int(yte_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS BiLSTM | 4s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión BiLSTM 4s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — BiLSTM 4s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

#crnn

##10 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, GRU, Conv1D,
                                     MaxPooling1D, GlobalAveragePooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  CRNN — Convolutional + Recurrent Neural Network
#  Raw Waveform | Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 10                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file_crnn(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(X_files, y_files):
    files_0, files_1 = X_files[y_files == 0], X_files[y_files == 1]
    diff = len(files_0) - len(files_1)
    if diff > 0:
        idx = np.random.choice(len(files_1), size=diff, replace=True)
        return (np.concatenate([X_files, files_1[idx]]),
                np.concatenate([y_files, np.ones(diff, dtype=int)]))
    return X_files, y_files

def build_dataset(file_list, label_list):
    X_all, y_all = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file_crnn(fp, label_list[i])
        X_all.extend(sx); y_all.extend(sy)
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)

# --- 4. MODELO CRNN ---
def get_model_crnn(input_length):
    """
    CRNN adaptado para segmentos de 2s (44100 muestras).
    Strides ajustados para mantener timesteps razonables en las GRU.
    44100 → Conv(stride=128)→~345 → Pool(4)→~86 → Pool(4)→~21
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque CNN 1: envolvente/periodicidad — stride agresivo para 2s
    x = Conv1D(64, kernel_size=128, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque CNN 2: estructuras fonéticas
    x = Conv1D(128, kernel_size=16, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.25)(x)

    # Bloque CNN 3: características finas
    x = Conv1D(256, kernel_size=8, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque CNN 4: abstracción
    x = Conv1D(256, kernel_size=4, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque BiGRU
    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(x)
    x = Dropout(0.3)(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CRNN_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION CRNN | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    X_tr_f, X_te_f = all_files[train_idx], all_files[test_idx]
    y_tr_f, y_te_f = all_labels[train_idx], all_labels[test_idx]
    X_tr_f, y_tr_f = oversample_minority(X_tr_f, y_tr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(X_tr_f, y_tr_f)
    print(f"  Shape: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_crnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=callbacks, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(X_te_f):
        segs_X, _ = process_file_crnn(fp, y_te_f[i])
        if not segs_X: continue
        segs_X = np.array(segs_X, dtype=np.float32)
        avg_prob = float(np.mean(model.predict(segs_X, verbose=0)))
        global_y_true.append(int(y_te_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS CRNN | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión CRNN 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — CRNN 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##2 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, GRU, Conv1D,
                                     MaxPooling1D, GlobalAveragePooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  CRNN — Convolutional + Recurrent Neural Network
#  Raw Waveform | Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 2                                      # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file_crnn(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(X_files, y_files):
    files_0, files_1 = X_files[y_files == 0], X_files[y_files == 1]
    diff = len(files_0) - len(files_1)
    if diff > 0:
        idx = np.random.choice(len(files_1), size=diff, replace=True)
        return (np.concatenate([X_files, files_1[idx]]),
                np.concatenate([y_files, np.ones(diff, dtype=int)]))
    return X_files, y_files

def build_dataset(file_list, label_list):
    X_all, y_all = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file_crnn(fp, label_list[i])
        X_all.extend(sx); y_all.extend(sy)
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)

# --- 4. MODELO CRNN ---
def get_model_crnn(input_length):
    """
    CRNN adaptado para segmentos de 2s (44100 muestras).
    Strides ajustados para mantener timesteps razonables en las GRU.
    44100 → Conv(stride=128)→~345 → Pool(4)→~86 → Pool(4)→~21
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque CNN 1: envolvente/periodicidad — stride agresivo para 2s
    x = Conv1D(64, kernel_size=128, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque CNN 2: estructuras fonéticas
    x = Conv1D(128, kernel_size=16, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.25)(x)

    # Bloque CNN 3: características finas
    x = Conv1D(256, kernel_size=8, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque CNN 4: abstracción
    x = Conv1D(256, kernel_size=4, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque BiGRU
    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(x)
    x = Dropout(0.3)(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CRNN_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION CRNN | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    X_tr_f, X_te_f = all_files[train_idx], all_files[test_idx]
    y_tr_f, y_te_f = all_labels[train_idx], all_labels[test_idx]
    X_tr_f, y_tr_f = oversample_minority(X_tr_f, y_tr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(X_tr_f, y_tr_f)
    print(f"  Shape: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_crnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=callbacks, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(X_te_f):
        segs_X, _ = process_file_crnn(fp, y_te_f[i])
        if not segs_X: continue
        segs_X = np.array(segs_X, dtype=np.float32)
        avg_prob = float(np.mean(model.predict(segs_X, verbose=0)))
        global_y_true.append(int(y_te_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS CRNN | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión CRNN 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — CRNN 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##3 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, GRU, Conv1D,
                                     MaxPooling1D, GlobalAveragePooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  CRNN — Convolutional + Recurrent Neural Network
#  Raw Waveform | Segmentos de 3 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 3                                      # ← 3 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 66 150
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 33 075
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file_crnn(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(X_files, y_files):
    files_0, files_1 = X_files[y_files == 0], X_files[y_files == 1]
    diff = len(files_0) - len(files_1)
    if diff > 0:
        idx = np.random.choice(len(files_1), size=diff, replace=True)
        return (np.concatenate([X_files, files_1[idx]]),
                np.concatenate([y_files, np.ones(diff, dtype=int)]))
    return X_files, y_files

def build_dataset(file_list, label_list):
    X_all, y_all = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file_crnn(fp, label_list[i])
        X_all.extend(sx); y_all.extend(sy)
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)

# --- 4. MODELO CRNN ---
def get_model_crnn(input_length):
    """
    CRNN adaptado para segmentos de 3s (66150 muestras).
    66150 → Conv(stride=256)→~259 → Pool(4)→~65 → Pool(4)→~16
    Secuencia manejable para las capas BiGRU.
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque CNN 1
    x = Conv1D(64, kernel_size=512, strides=256, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque CNN 2
    x = Conv1D(128, kernel_size=32, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.25)(x)

    # Bloque CNN 3
    x = Conv1D(256, kernel_size=8, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque CNN 4
    x = Conv1D(256, kernel_size=4, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque BiGRU
    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(x)
    x = Dropout(0.3)(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CRNN_3s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION CRNN | 3s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    X_tr_f, X_te_f = all_files[train_idx], all_files[test_idx]
    y_tr_f, y_te_f = all_labels[train_idx], all_labels[test_idx]
    X_tr_f, y_tr_f = oversample_minority(X_tr_f, y_tr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(X_tr_f, y_tr_f)
    print(f"  Shape: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_crnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=callbacks, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(X_te_f):
        segs_X, _ = process_file_crnn(fp, y_te_f[i])
        if not segs_X: continue
        segs_X = np.array(segs_X, dtype=np.float32)
        avg_prob = float(np.mean(model.predict(segs_X, verbose=0)))
        global_y_true.append(int(y_te_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS CRNN | 3s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión CRNN 3s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — CRNN 3s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

##4 s

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from sklearn.utils import class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Bidirectional, GRU, Conv1D,
                                     MaxPooling1D, GlobalAveragePooling1D, Input)

# ══════════════════════════════════════════════════════════════
#  CRNN — Convolutional + Recurrent Neural Network
#  Raw Waveform | Segmentos de 2 segundos | Overlap 50%
# ══════════════════════════════════════════════════════════════

# --- 1. CONFIGURACIÓN ---
BASE_PATH_0 = r'wav/0'
BASE_PATH_1 = r'wav/1'

SAMPLE_RATE         = 22050
SEGMENT_DURATION    = 4                                     # ← 2 segundos
SAMPLES_PER_SEGMENT = int(SEGMENT_DURATION * SAMPLE_RATE)   # 44 100
HOP_SIZE            = SAMPLES_PER_SEGMENT // 2              # 50% overlap → 22 050
N_FOLDS             = 5

# --- 2. LISTAS DE ARCHIVOS ---
def get_file_lists():
    files_0 = [os.path.join(BASE_PATH_0, f)
               for f in os.listdir(BASE_PATH_0) if f.endswith('.wav')]
    files_1 = [os.path.join(BASE_PATH_1, f)
               for f in os.listdir(BASE_PATH_1) if f.endswith('.wav')]
    return (np.array(files_0 + files_1),
            np.array([0]*len(files_0) + [1]*len(files_1)))

all_files, all_labels = get_file_lists()
print(f"Total archivos: {len(all_files)}")
print(f" - Sin Depresión (0): {np.sum(all_labels == 0)}")
print(f" - Con Depresión (1): {np.sum(all_labels == 1)}")
print(f"Segment: {SEGMENT_DURATION}s | Samples: {SAMPLES_PER_SEGMENT} | Hop: {HOP_SIZE} (50% overlap)")

# --- 3. PROCESAMIENTO CON OVERLAP ---
def process_file_crnn(file_path, label):
    segments_X, segments_y = [], []
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        for start in range(0, len(audio) - SAMPLES_PER_SEGMENT + 1, HOP_SIZE):
            seg = audio[start:start + SAMPLES_PER_SEGMENT]
            if len(seg) < SAMPLES_PER_SEGMENT:
                seg = np.pad(seg, (0, SAMPLES_PER_SEGMENT - len(seg)))
            seg = seg / (np.max(np.abs(seg)) + 1e-8)
            segments_X.append(seg.reshape(-1, 1))
            segments_y.append(label)
    except Exception as e:
        print(f"Error {file_path}: {e}")
    return segments_X, segments_y

def oversample_minority(X_files, y_files):
    files_0, files_1 = X_files[y_files == 0], X_files[y_files == 1]
    diff = len(files_0) - len(files_1)
    if diff > 0:
        idx = np.random.choice(len(files_1), size=diff, replace=True)
        return (np.concatenate([X_files, files_1[idx]]),
                np.concatenate([y_files, np.ones(diff, dtype=int)]))
    return X_files, y_files

def build_dataset(file_list, label_list):
    X_all, y_all = [], []
    for i, fp in enumerate(file_list):
        sx, sy = process_file_crnn(fp, label_list[i])
        X_all.extend(sx); y_all.extend(sy)
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)

# --- 4. MODELO CRNN ---
def get_model_crnn(input_length):
    """
    CRNN adaptado para segmentos de 2s (44100 muestras).
    Strides ajustados para mantener timesteps razonables en las GRU.
    44100 → Conv(stride=128)→~345 → Pool(4)→~86 → Pool(4)→~21
    """
    inputs = Input(shape=(input_length, 1))

    # Bloque CNN 1: envolvente/periodicidad — stride agresivo para 2s
    x = Conv1D(64, kernel_size=128, strides=64, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4)(x)
    x = Dropout(0.2)(x)

    # Bloque CNN 2: estructuras fonéticas
    x = Conv1D(128, kernel_size=16, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.25)(x)

    # Bloque CNN 3: características finas
    x = Conv1D(256, kernel_size=8, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque CNN 4: abstracción
    x = Conv1D(256, kernel_size=4, strides=1, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Bloque BiGRU
    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(x)
    x = Dropout(0.3)(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='CRNN_2s_Overlap50')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- 5. CROSS-VALIDATION ---
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
global_y_true, global_y_pred, global_y_prob = [], [], []

print(f"\nINICIANDO CROSS-VALIDATION CRNN | 2s | Overlap 50% ({N_FOLDS} FOLDS)\n")

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_files, all_labels)):
    print(f"\n--- FOLD {fold_idx + 1}/{N_FOLDS} ---")
    X_tr_f, X_te_f = all_files[train_idx], all_files[test_idx]
    y_tr_f, y_te_f = all_labels[train_idx], all_labels[test_idx]
    X_tr_f, y_tr_f = oversample_minority(X_tr_f, y_tr_f)

    print("Generando segmentos...")
    X_train, y_train = build_dataset(X_tr_f, y_tr_f)
    print(f"  Shape: {X_train.shape}")

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw = dict(enumerate(weights))

    model = get_model_crnn(SAMPLES_PER_SEGMENT)
    if fold_idx == 0:
        model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-7)
    ]
    model.fit(X_train, y_train, epochs=25, batch_size=32,
              class_weight=cw, callbacks=callbacks, verbose=1)
    print("Entrenamiento completado.")

    for i, fp in enumerate(X_te_f):
        segs_X, _ = process_file_crnn(fp, y_te_f[i])
        if not segs_X: continue
        segs_X = np.array(segs_X, dtype=np.float32)
        avg_prob = float(np.mean(model.predict(segs_X, verbose=0)))
        global_y_true.append(int(y_te_f[i]))
        global_y_pred.append(1 if avg_prob > 0.33 else 0)
        global_y_prob.append(avg_prob)

    tf.keras.backend.clear_session()
    del model, X_train, y_train

# --- 6. REPORTE FINAL ---
print("\n" + "="*60)
print(f"RESULTADOS CRNN | 2s | Overlap 50%  ({len(global_y_true)} pacientes)")
print("="*60)

cm = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Depresión', 'Depresión'],
            yticklabels=['No Depresión', 'Depresión'])
plt.title(f'Confusión CRNN 2s 50% ({N_FOLDS}-Fold CV)')
plt.ylabel('Real'); plt.xlabel('Predicción')
plt.tight_layout(); plt.show()

acc      = accuracy_score(global_y_true, global_y_pred)
prec     = precision_score(global_y_true, global_y_pred, zero_division=0)
rec      = recall_score(global_y_true, global_y_pred, zero_division=0)
f1       = f1_score(global_y_true, global_y_pred, zero_division=0)
f1_macro = f1_score(global_y_true, global_y_pred, average='macro', zero_division=0)
print(f"Accuracy:   {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Dep:     {f1:.4f}  | F1-Macro:  {f1_macro:.4f}")
print(classification_report(global_y_true, global_y_pred,
                             target_names=['No Depresión', 'Depresión']))

fpr, tpr, _ = roc_curve(global_y_true, global_y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'navy', lw=2, linestyle='--')
plt.xlabel('Falsos Positivos'); plt.ylabel('Verdaderos Positivos')
plt.title('ROC — CRNN 2s Overlap 50%')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

#Transformer1d

In [ ]:
pip install vit-keras

In [ ]:
pip install transformers

##10 s

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_mfcc"   # "transformer_mfcc" | "transformer_waveform"
SEG_SECONDS = 10.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_waveform"
SEG_SECONDS = 10.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

##2 s

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_mfcc"   # "transformer_mfcc" | "transformer_waveform"
SEG_SECONDS = 2.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  =  "transformer_waveform"
SEG_SECONDS = 2.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

##3 s

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_mfcc"   # "transformer_mfcc" | "transformer_waveform"
SEG_SECONDS = 3.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_waveform"
SEG_SECONDS = 3.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

##4 s

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_mfcc"   # "transformer_mfcc" | "transformer_waveform"
SEG_SECONDS = 4.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")

In [ ]:
import numpy as np
import librosa
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# CONFIG
# =============================================================================

# ── CAMBIA ESTO según tu estructura ──────────────────────────────────────────
# Opción A: wav/0  y  /wav/1
DATA_ROOT = Path("wav")

# Opción B: Depresión y Ansiedad/Ansiedad/0  y  /1
# DATA_ROOT = Path("Depresión y Ansiedad/Ansiedad")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_TYPE  = "transformer_waveform"
SEG_SECONDS = 4.0

SR          = 16000
HOP_SECONDS = SEG_SECONDS * 0.5
N_MFCC      = 40
HOP_LENGTH  = 512
N_FFT       = 1024
N_FEATURES  = N_MFCC * 3

SEG_SAMPLES     = int(SR * SEG_SECONDS)
HOP_SAMPLES     = int(SR * HOP_SECONDS)
EXPECTED_FRAMES = int(np.ceil(SEG_SAMPLES / HOP_LENGTH))

D_MODEL    = 64
NUM_HEADS  = 4
FF_DIM     = 128
NUM_BLOCKS = 2
DROPOUT_TR = 0.1

BATCH_SIZE      = 32
EPOCHS          = 40
N_FOLDS         = 5
INNER_VAL_FRAC  = 0.2
STEPS_PER_EPOCH = 200
VAL_STEPS       = 60

print("=" * 60)
print(f"  PIPELINE TRANSFORMER 1D — SIN SPLIT PREVIO")
print(f"  Modelo  : {MODEL_TYPE}")
print(f"  Segmento: {SEG_SECONDS}s  |  Hop: {HOP_SECONDS}s (50 % overlap)")
print(f"  Datos   : {DATA_ROOT}")
print("=" * 60)

# =============================================================================
# 1. CARGA DE ARCHIVOS  ← ÚNICO CAMBIO RESPECTO AL ORIGINAL
#    Lee directamente DATA_ROOT/0/*.wav  y  DATA_ROOT/1/*.wav
#    No hay separación train/test previa; el CV la hace internamente.
# =============================================================================

def list_wavs(root: Path):
    files, labels = [], []
    for cls in ["0", "1"]:
        d = root / cls
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        wavs = sorted(d.glob("*.wav"))
        print(f"  Clase {cls}: {len(wavs)} archivos en {d}")
        for p in wavs:
            files.append(str(p))
            labels.append(int(cls))
    return np.array(files), np.array(labels, dtype=np.int32)


all_files, all_labels = list_wavs(DATA_ROOT)

print(f"\nTotal audios : {len(all_files)}")
print(f"  Clase 0    : {int((all_labels == 0).sum())}")
print(f"  Clase 1    : {int((all_labels == 1).sum())}")

if len(all_files) == 0:
    raise RuntimeError(
        f"No se encontraron archivos .wav en {DATA_ROOT}/0 o {DATA_ROOT}/1\n"
        f"Verifica que la ruta DATA_ROOT sea correcta."
    )

# =============================================================================
# 2. EXTRACCIÓN DE SEGMENTOS  (idéntica al original)
# =============================================================================

def audio_to_raw_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    mx = np.max(np.abs(y))
    if mx > 0: y = y / mx
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        segs.append(y[start:start + SEG_SAMPLES].astype(np.float32))
    if not segs:
        segs.append(y[:SEG_SAMPLES].astype(np.float32))
    X  = np.stack(segs)[..., np.newaxis]
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def audio_to_mfcc_segments(wav_path, label):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < SEG_SAMPLES:
        y = np.pad(y, (0, SEG_SAMPLES - len(y)))
    segs = []
    for start in range(0, len(y) - SEG_SAMPLES + 1, HOP_SAMPLES):
        seg  = y[start:start + SEG_SAMPLES]
        mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        d1   = librosa.feature.delta(mfcc)
        d2   = librosa.feature.delta(mfcc, order=2)
        feat = np.concatenate([mfcc, d1, d2], axis=0).T
        feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        if feat.shape[0] < EXPECTED_FRAMES:
            feat = np.pad(feat, ((0, EXPECTED_FRAMES - feat.shape[0]), (0, 0)))
        else:
            feat = feat[:EXPECTED_FRAMES]
        segs.append(feat.astype(np.float32))
    if not segs:
        segs.append(np.zeros((EXPECTED_FRAMES, N_FEATURES), dtype=np.float32))
    X  = np.stack(segs)
    ys = np.full(X.shape[0], int(label), dtype=np.int32)
    return X, ys


def extract_segments(wav_path, label):
    return (audio_to_raw_segments(wav_path, label)
            if MODEL_TYPE == "transformer_waveform"
            else audio_to_mfcc_segments(wav_path, label))


def get_input_shape():
    return ((SEG_SAMPLES, 1)
            if MODEL_TYPE == "transformer_waveform"
            else (EXPECTED_FRAMES, N_FEATURES))

INPUT_SHAPE = get_input_shape()

# =============================================================================
# 3. tf.data  (idéntico al original)
# =============================================================================

def build_dataset(file_paths, labels, batch_size=32, shuffle=True, repeat=True):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i]
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if repeat: ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)


def build_dataset_with_fp(file_paths, labels, batch_size=32):
    def gen():
        for fp, lab in zip(file_paths, labels):
            X, y_seg = extract_segments(fp, int(lab))
            for i in range(X.shape[0]):
                yield X[i], y_seg[i], fp
    sig = (tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.string))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.batch(batch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def count_segments(file_paths, labels):
    n0 = n1 = 0
    for fp, lab in zip(file_paths, labels):
        X, _ = extract_segments(fp, int(lab))
        if int(lab) == 0: n0 += X.shape[0]
        else:              n1 += X.shape[0]
    return n0, n1

# =============================================================================
# 4. MODELOS  (idénticos al original)
# =============================================================================

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att   = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn   = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.ln1(x + attn)
        ffn  = self.ffn(x)
        ffn  = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)


def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims      = np.arange(depth)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / depth)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], tf.float32)


def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    residual = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1)(residual)
    return layers.Add()([x, residual])


def build_transformer_waveform():
    inp = layers.Input(shape=(SEG_SAMPLES, 1))
    x   = layers.Conv1D(32, 80, strides=4, padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)
    x   = layers.MaxPool1D(4)(x)
    x   = layers.Dropout(0.15)(x)
    for dilation in [1, 2, 4, 8]:
        x = tcn_block(x, 64, 3, dilation, 0.2)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(4)(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer_mfcc():
    inp = layers.Input(shape=(EXPECTED_FRAMES, N_FEATURES))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(D_MODEL)(x)
    seq_len = x.shape[1]
    if seq_len is not None:
        x = x + positional_encoding(seq_len, D_MODEL)
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT_TR)(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
    return m


def make_model():
    return (build_transformer_waveform()
            if MODEL_TYPE == "transformer_waveform"
            else build_transformer_mfcc())

# =============================================================================
# 5. EVALUACIÓN  (idéntica al original)
# =============================================================================

def predict_by_audio(model, ds_with_fp):
    probs_by_file = {}
    label_by_file = {}
    for Xb, yb, fpb in ds_with_fp:
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for p, y, fp in zip(pb, yb.numpy(), fpb.numpy()):
            fp = fp.decode("utf-8")
            probs_by_file.setdefault(fp, []).append(float(p))
            label_by_file[fp] = int(y)
    files  = sorted(probs_by_file.keys())
    y_true = np.array([label_by_file[f]          for f in files], dtype=int)
    y_prob = np.array([np.mean(probs_by_file[f])  for f in files], dtype=float)
    return files, y_true, y_prob


def pick_best_threshold(y_true, y_prob):
    best = None
    for t in np.linspace(0.05, 0.95, 181):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if best is None or f1 > best["f1"]:
            best = {"t": float(t), "f1": float(f1),
                    "prec": float(precision_score(y_true, y_pred, zero_division=0)),
                    "rec":  float(recall_score(y_true, y_pred,  zero_division=0)),
                    "pos":  int(y_pred.sum())}
    return best

# =============================================================================
# 6. CROSS-VALIDATION 5-FOLD OOF  (lógica idéntica, sin split previo)
# =============================================================================

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_prob = np.zeros(len(all_files), dtype=float)
oof_pred = np.zeros(len(all_files), dtype=int)
oof_true = all_labels.copy()

print(f"\n🚀 Iniciando {N_FOLDS}-Fold CV  |  {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_files, all_labels), start=1):

    print(f"\n{'─'*55}\n  FOLD {fold}/{N_FOLDS}\n{'─'*55}")

    fold_tr_files  = all_files[train_idx]
    fold_tr_labels = all_labels[train_idx]
    fold_te_files  = all_files[test_idx]
    fold_te_labels = all_labels[test_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                 random_state=fold + 100)
    tr2_idx, val2_idx = next(sss.split(fold_tr_files, fold_tr_labels))
    tr2_files  = fold_tr_files[tr2_idx]
    tr2_labels = fold_tr_labels[tr2_idx]
    val2_files = fold_tr_files[val2_idx]
    val2_labels= fold_tr_labels[val2_idx]

    n0, n1 = count_segments(tr2_files, tr2_labels)
    total  = n0 + n1
    cw     = {0: total / (2 * n0), 1: total / (2 * n1)} if (n0 > 0 and n1 > 0) else None
    print(f"  Segmentos sub-train → clase 0: {n0} | clase 1: {n1}")
    print(f"  Class weights: {cw}")

    train_ds     = build_dataset(tr2_files,   tr2_labels,   BATCH_SIZE, shuffle=True,  repeat=True)
    val_ds       = build_dataset(val2_files,  val2_labels,  BATCH_SIZE, shuffle=False, repeat=True)
    val_eval_ds  = build_dataset_with_fp(val2_files,    val2_labels,   BATCH_SIZE)
    test_eval_ds = build_dataset_with_fp(fold_te_files, fold_te_labels, BATCH_SIZE)

    model = make_model()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=0),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VAL_STEPS,
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )

    _, yv_true, yv_prob = predict_by_audio(model, val_eval_ds)
    best = pick_best_threshold(yv_true, yv_prob)
    print(f"  Threshold elegido en val: {best}")

    files_fold, yt_true, yt_prob = predict_by_audio(model, test_eval_ds)
    yt_pred = (yt_prob >= best["t"]).astype(int)

    path_to_res = {fp: (float(p), int(pr))
                   for fp, p, pr in zip(files_fold, yt_prob, yt_pred)}
    for idx in test_idx:
        oof_prob[idx], oof_pred[idx] = path_to_res[all_files[idx]]

    tf.keras.backend.clear_session()
    del model

# =============================================================================
# 7. REPORTE GLOBAL OOF
# =============================================================================

print("\n" + "=" * 60)
print(f"  RESULTADOS GLOBALES OOF ({len(oof_true)} audios)")
print(f"  Modelo: {MODEL_TYPE}  |  Segmento: {SEG_SECONDS}s")
print("=" * 60)

print(classification_report(oof_true, oof_pred,
      target_names=["Clase 0", "Clase 1"], zero_division=0))

cm = confusion_matrix(oof_true, oof_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clase 0", "Clase 1"],
            yticklabels=["Clase 0", "Clase 1"])
plt.title(f"Confusión OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(oof_true, oof_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC OOF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1.5, color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"Curva ROC OOF — {MODEL_TYPE} | {SEG_SECONDS}s")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"AUC OOF          : {roc_auc:.4f}")
print(f"Prob min/mean/max: {oof_prob.min():.3f} / "
      f"{oof_prob.mean():.3f} / {oof_prob.max():.3f}")
print(f"Predichos clase 1: {int(oof_pred.sum())} / {len(oof_pred)}")
print(f"Reales  clase 1  : {int(oof_true.sum())} / {len(oof_true)}")